# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id` fields. Follow through each section to understand how to programmatically access record sets, fields, columns, and perform initial exploratory analysis.

### Dataset Source
The dataset ("FAIR^2" package) Croissant schema is available at:

**https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json**

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and access its properties using `mlcroissant`. The metadata object provides schema-defined properties documented in the Croissant file.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print basic dataset-level metadata properties
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Citation: {dataset.metadata.citeAs}")
print(f"License: {dataset.metadata.license}")
print(f"Conforms to: {dataset.metadata.conformsTo}")

## 2. Data Overview
Explore available record sets, along with their fields and respective `@id` values. All Croissant entities (record sets, fields, columns) are referenced by their `@id` values.

Below, we list all main record sets, and, for each, summarize their available fields and columns.

In [ ]:
print("Available Record Sets (by @id):")
for rs in dataset.record_sets:
    print(f"  - {rs['@id']}")
    print(f"    Name: {rs.get('name', '(no name)')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # Single field dict
        fields = [fields]
    if fields:
        print("    Fields (by @id):")
        for f in fields:
            if isinstance(f, str):
                print(f"      - {f}")
            elif isinstance(f, dict):
                print(f"      - {f.get('@id')} (name: {f.get('name', '')})")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print("    Columns (by @id):")
        for c in columns:
            if isinstance(c, str):
                print(f"      - {c}")
            elif isinstance(c, dict):
                print(f"      - {c.get('@id')} (name: {c.get('name', '')})")
    print()
print(f"Total record sets: {len(dataset.record_sets)}")

### List first records in a record set
Demonstrate how to use the record set's `@id` to programmatically access and print the first records. Choose an available record set for demonstration below.

In [ ]:
# Pick the main record set (by @id)
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
if len(record_sets_ids) > 0:
    main_record_set_id = record_sets_ids[0]
    print(f"First few records from record set: {main_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(record)
        if i > 2:
            break
else:
    print('No record sets were defined in the dataset schema. Please check the Croissant metadata.')

## 3. Data Extraction
Load each record set (referenced by its `@id`) into a Pandas DataFrame for further analysis. This step creates a mapping of record set `@id` to DataFrame, containing all corresponding data rows.

In [ ]:
dataframes = {}
if dataset.record_sets:
    for rs in dataset.record_sets:
        rs_id = rs['@id']
        print(f"Loading records for record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame.from_records(records)
            dataframes[rs_id] = df
            print(f"  - Loaded {len(df)} rows; columns: {df.columns.tolist()}")
        else:
            print("  - No records found.")
else:
    print('No record sets were found.')

In [ ]:
# Show the first few rows of the main DataFrame (if available)
if dataframes:
    # Pick main record set (reuse main_record_set_id if defined)
    main_rs = None
    if 'main_record_set_id' in locals():
        main_rs = main_record_set_id
    else:
        main_rs = list(dataframes.keys())[0]
    print(f"Columns in {main_rs}:")
    print(dataframes[main_rs].columns.tolist())
    dataframes[main_rs].head()
else:
    print('No dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply a few standard EDA steps. All references to fields are made by their schema `@id`. You can use a numeric field for filtering, normalization, or grouping.

Below, try for a field like `cr:coverage_percentage` or another available numeric field (replace the example field IDs with those enumerated in section 2 as needed).

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

if dataframes:
    # Use main record set loaded above
    rs_id = main_rs
    df = dataframes[rs_id].copy()
    # Attempt to choose a numeric field by checking dtypes, prefer those with 'coverage' or 'count' in the name
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]
    if not numeric_candidates:
        # Fallback: try columns with likely numeric field names
        for col in df.columns:
            if any(keyword in col.lower() for keyword in ['coverage', 'count', 'score', 'abundance', 'weight', 'mw']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    pass
        numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]

    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Analyzing numeric field (by @id/column): {numeric_field}")
        threshold = df[numeric_field].quantile(0.5)  # Use median as threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a field such as 'cr:description' or any with <15 unique values
        group_candidates = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < 15]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouping by field (by @id/column): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields were detected for EDA.")
else:
    print('No dataframes available for EDA analysis.')

## 5. Visualization
Visualize the distribution of a selected numeric field (by `@id` or column name) and optionally show relationships with another group/categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If grouping field exists, do a boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(10,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
- We demonstrated how to explore a Croissant-structured dataset using `mlcroissant`, including explicit referencing of all entities via their `@id` values.
- Record sets, fields, and columns were programmatically discovered and loaded.
- Numeric fields were analyzed and visualized, with EDA steps such as filtering, normalization, and grouping, all by schema-defined `@id`.

Further analyses can be built atop this foundation for downstream applications.